### Data

In [1]:
import torch
import pandas as pd

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(train.shape)
print(test.shape)
train.head(10)

(7613, 5)
(3263, 4)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1
8,14,NaN,NaN,There's an emergency evacuation happening now ...,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our a...,1


In [4]:
test.head(10)

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
5,12,NaN,NaN,We're shaking...It's an earthquake
6,21,NaN,NaN,They'd probably still show more life than Arse...
7,22,NaN,NaN,Hey! How are you?
8,27,NaN,NaN,What a nice hat?
9,29,NaN,NaN,Fuck off!


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3263 entries, 0 to 3262
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        3263 non-null   int64 
 1   keyword   3237 non-null   object
 2   location  2158 non-null   object
 3   text      3263 non-null   object
dtypes: int64(1), object(3)
memory usage: 102.1+ KB


In [7]:
train.isnull().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [8]:
test.isnull().sum()

id             0
keyword       26
location    1105
text           0
dtype: int64

In [9]:
train['keyword'] = train['keyword'].fillna('none')
test['keyword'] = test['keyword'].fillna('none')

In [10]:
train['location'] = train['location'].fillna('unknown')
test['location'] = test['location'].fillna('unknown')

In [11]:
train.isnull().sum()

id          0
keyword     0
location    0
text        0
target      0
dtype: int64

In [12]:
test.isnull().sum()

id          0
keyword     0
location    0
text        0
dtype: int64

In [13]:
# Combine keyword + text
train['text_combined'] = train['keyword'] + ' ' + train['text']
test['text_combined'] = test['keyword'] + ' ' + test['text']

In [14]:
#Text normalization
import re

def normalize(text):
    text = text.lower()                        # lowercase
    text = re.sub(r'http\S+|www\S+', '', text) # URL বাদ
    text = re.sub(r'@\w+', '', text)           # @mention বাদ
    text = re.sub(r'#', '', text)               # শুধু '#' চিহ্ন বাদ, hashtag word রাখলাম
    text = re.sub(r'[^a-z0-9\s]', '', text)     # punctuation, special char বাদ
    text = ' '.join(text.split())                # extra whitespace বাদ
    return text

In [15]:
train['text_clean'] = train['text'].apply(normalize)
test['text_clean'] = test['text'].apply(normalize)

In [16]:
print("\nBefore:", train['text'].iloc[0])
print("After :", train['text_clean'].iloc[0])


Before: Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
After : our deeds are the reason of this earthquake may allah forgive us all


In [17]:
# keyword + cleaned text pare (Combine)
train['text_combined'] = train['keyword'] + ' ' + train['text_clean']
test['text_combined'] = test['keyword'] + ' ' + test['text_clean']

train[['keyword', 'text', 'text_clean', 'text_combined']].head()

,keyword,text,text_clean,text_combined
0,none,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,none our deeds are the reason of this earthqua...
1,none,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,none forest fire near la ronge sask canada
2,none,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...,none all residents asked to shelter in place a...
3,none,"13,000 people receive #wildfires evacuation or...",13000 people receive wildfires evacuation orde...,none 13000 people receive wildfires evacuation...
4,none,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,none just got sent this photo from ruby alaska...


### Model

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [19]:
#Feature and target

X = train['text_combined']
y = train['target']

In [20]:
# Train/Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)

Train size: (6090,)
Validation size: (1523,)


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

#TF-IDF Vectorizer make
vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)

# Train data fit + transform
X_train_vec = vectorizer.fit_transform(X_train)

# Only transform (do not fit!) the validation data.
X_val_vec = vectorizer.transform(X_val)

print("X_train_vec shape:", X_train_vec.shape)
print("X_val_vec shape:", X_val_vec.shape)

X_train_vec shape: (6090, 9431)
X_val_vec shape: (1523, 9431)


In [23]:
from sklearn.linear_model import LogisticRegression

# Model creation (not yet trained, only the object has been created)
model = LogisticRegression(max_iter=1000)

print(model)

LogisticRegression(max_iter=1000)


### Training

In [24]:
model.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [25]:
from sklearn.metrics import f1_score, classification_report

# Making predictions on validation data
val_preds = model.predict(X_val_vec)

# Measuring F1 Score (Kaggle's evaluation metric)
f1 = f1_score(y_val, val_preds)
print("Validation F1 Score:", f1)

# Detailed report (precision, recall, etc.)
print("\nClassification Report:\n")
print(classification_report(y_val, val_preds))

Validation F1 Score: 0.7449062754686226

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.86      0.83       874
           1       0.79      0.70      0.74       649

    accuracy                           0.79      1523
   macro avg       0.79      0.78      0.79      1523
weighted avg       0.79      0.79      0.79      1523



In [26]:
# Re-vectorize the entire training dataset (using all data, without excluding the validation set)
X_full_vec = vectorizer.fit_transform(train['text_combined'])
test_vec = vectorizer.transform(test['text_combined'])

# New model, trained on the full dataset
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_full_vec, train['target'])

# Predict on test data
test_preds = final_model.predict(test_vec)

print("Test predictions done. Total:", len(test_preds))

Test predictions done. Total: 3263


In [27]:
# Submission file make
submission = pd.DataFrame({
    'id': test['id'],
    'target': test_preds
})

# Saving to a CSV file
submission.to_csv('submission.csv', index=False)

print("submission.csv has been created. Shape:", submission.shape)
submission.head()

submission.csv has been created. Shape: (3263, 2)


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
